# Fashion-MNIST: เปรียบเทียบ DNN, CNN และทดลองปรับปรุงโมเดล
รันเซลล์จากบนลงล่าง หรือ Restart Kernel แล้ว Run All เพื่อให้ตัวแปรตรงกันทั้งงาน

ลำดับงาน: ตรวจข้อมูล → แบ่ง Train/Validation → ฝึก DNN โมเดลเดิม และ CNN → เลือก โมเดลเดิมที่เลือก ด้วย Validation → เพิ่ม Dropout, L2 และ Early Stopping → เปรียบเทียบ → เลือกโมเดลสุดท้าย → ประเมิน Test → วิเคราะห์ภาพที่ผิด

**บทบาทของข้อมูล:** Training ใช้ปรับน้ำหนัก, Validation ใช้เลือกและปรับโมเดล, Test ใช้ประเมินท้ายงาน ห้ามนำคะแนน Test กลับไปจูน

ไฟล์ฉบับเรียบเรียงนี้ล้างผลรันเก่าเพื่อไม่ให้ปะปนกับโค้ดใหม่ ต้องรันเพื่อสร้างผลจริง ไม่มีการใส่คะแนนสมมติ

## ชื่อโมเดลที่ใช้ตลอดไฟล์
| ชื่อที่แสดงในผลและกราฟ | ความหมาย | ตัวแปรโมเดล |
|---|---|---|
| DNN Original | DNN โมเดลเดิม ก่อนเพิ่มเทคนิค | `dnn_original` |
| CNN Original | CNN โมเดลเดิม ก่อนเพิ่มเทคนิค | `cnn_original` |
| CNN New (Dropout + L2 + Early Stopping) หรือ DNN New (Dropout + L2 + Early Stopping) | โมเดลใหม่ สร้างตามชนิดของโมเดลเดิมที่เลือกและเพิ่มสามเทคนิค | `new_model` |

`baseline` อ้างถึงโมเดลเดิมที่เลือก ไม่ใช่โมเดลเพิ่มอีกตัว ส่วน `final_model` อ้างถึงตัวที่เลือกใช้หลังเปรียบเทียบ ซึ่งอาจเป็นโมเดลเดิมหรือโมเดลใหม่ก็ได้ ชื่อกราฟใช้ Original = เดิม และ New = ใหม่ เพื่ออ่านได้แม้เครื่องไม่มีฟอนต์ไทย


## 1. เตรียมไลบรารีและค่าการทดลอง
ถ้ายังไม่มีไลบรารี ให้เอาเครื่องหมาย `#` หน้า `%pip` ออกแล้วรัน จากนั้น Restart Kernel หากจำเป็น ค่าของ Seed ช่วยให้ทำซ้ำได้ แต่ผลอาจยังต่างตามอุปกรณ์และรุ่นไลบรารี


In [ ]:
# %pip install tensorflow numpy matplotlib scikit-learn pandas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from IPython.display import display

tf.keras.utils.set_random_seed(42)
EPOCHS = 10
BATCH_SIZE = 64


## 2. โหลดและตรวจข้อมูลดิบ
หนึ่งภาพมีขนาด 28 × 28 พิกเซล และ Label เป็นเลข 0–9 ตรวจจำนวนภาพให้ตรงกับจำนวนคำตอบก่อนใช้งาน ชุด Test แยกไว้ตั้งแต่ต้น


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = (
    tf.keras.datasets.fashion_mnist.load_data()
)

print("Training images:", x_train_full.shape)
print("Training labels:", y_train_full.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)


## 3. จับคู่เลขคลาสกับชื่อเสื้อผ้าและดูภาพตัวอย่าง
ชื่อใน `class_names` เรียงตามเลข Label ภาพต่อไปนี้เป็น 10 ภาพแรก ไม่จำเป็นต้องครบทั้ง 10 คลาส


In [ ]:
class_names = [
    "T-shirt/top",  # 0
    "Trouser",      # 1
    "Pullover",     # 2
    "Dress",        # 3
    "Coat",         # 4
    "Sandal",       # 5
    "Shirt",        # 6
    "Sneaker",      # 7
    "Bag",          # 8
    "Ankle boot"    # 9
]


In [ ]:
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i], cmap="gray")
    plt.title(class_names[y_train_full[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 4. ปรับพิกเซลเป็นช่วง 0–1
แปลงเป็น float32 แล้วหารด้วย 255 เพื่อให้ขนาดค่าข้อมูลเหมาะกับการฝึก ใช้ขั้นตอนเดียวกันกับ Train และ Test และอย่ารันเซลล์นี้ซ้ำโดยไม่โหลดข้อมูลดิบใหม่


In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Normalized pixel range:", x_train_full.min(), "to", x_train_full.max())


## 5. แบ่ง Validation จากชุด Training เดิม
แบ่ง 20% เป็น Validation และใช้ stratify รักษาสัดส่วนคลาส จึงได้ Train 48,000, Validation 12,000 และ Test 10,000 ภาพ ตารางช่วยตรวจจำนวนแต่ละคลาส


In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)


## 6. สร้าง DNN โมเดลเดิม
DNN รับภาพ (28, 28) แล้ว Flatten เป็นเวกเตอร์ 784 ค่า Dense เรียนรู้เพื่อทำนาย 10 คลาส ดู `summary()` เพื่อเชื่อมขนาดข้อมูลกับจำนวนพารามิเตอร์


In [ ]:
dnn_original = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
], name="dnn_original")

dnn_original.summary()


## 7. กำหนดวิธีเรียนรู้และฝึก DNN โมเดลเดิม
`compile()` ตั้ง Optimizer, Loss และตัวชี้วัด จากนั้น `fit()` ปรับน้ำหนักด้วย Training และตรวจ Validation หลังแต่ละ Epoch ใช้ sparse categorical crossentropy เพราะคำตอบเป็นเลขคลาส ไม่ใช่ one-hot


In [ ]:
dnn_original.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_dnn_original = dnn_original.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


## 8. ผลการฝึก DNN
`history_dnn_original` เก็บค่า Accuracy และ Loss ในแต่ละ Epoch เพื่อนำไปวาดกราฟและเปรียบเทียบโมเดล


In [ ]:
# history_dnn_original เก็บผล Accuracy และ Loss ของ DNN ในแต่ละ Epoch


## 9. เตรียมรูปทรงข้อมูลสำหรับ CNN
เพิ่มแกน Channel ขนาด 1 เพราะเป็นภาพสีเทา จาก (จำนวนภาพ, 28, 28) เป็น (จำนวนภาพ, 28, 28, 1) ภาพและคำตอบยังเป็นชุดเดียวกับ DNN


In [ ]:
x_train_cnn = x_train[..., np.newaxis]
x_val_cnn = x_val[..., np.newaxis]
x_test_cnn = x_test[..., np.newaxis]

for name, data in [("Train CNN", x_train_cnn), ("Validation CNN", x_val_cnn), ("Test CNN", x_test_cnn)]:
    print(f"{name}: {data.shape}")


## 10. สร้างและฝึก CNN โมเดลเดิม
Conv2D เรียนรู้ลักษณะเชิงพื้นที่ MaxPooling ลดขนาด Feature Map แล้ว Dense ใช้ลักษณะที่ได้จำแนกประเภท ใช้จำนวน Epoch และ Batch size เท่ากับ DNN


In [ ]:
cnn_original = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(
        32, (3, 3),
        activation="relu",
        padding="valid"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(
        64, (3, 3),
        activation="relu",
        padding="valid"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
], name="cnn_original")

cnn_original.summary()


In [ ]:
cnn_original.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_cnn_original = cnn_original.fit(
    x_train_cnn,
    y_train,
    validation_data=(x_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


## 11. เปรียบเทียบ Accuracy ของ DNN และ CNN
กราฟแสดง Training Accuracy และ Validation Accuracy ของโมเดลเดิมทั้งสองตัว ใช้เปรียบเทียบแนวโน้มระหว่างฝึก ส่วนกราฟ Loss ของโมเดลเดิมที่เลือกและโมเดลใหม่จะแสดงในข้อ 15


In [ ]:
# กราฟ Accuracy ของโมเดลเดิมทั้งสองตัว
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, history in zip(axes, ["DNN Original", "CNN Original"],
                              [history_dnn_original, history_cnn_original]):
    ax.plot(history.history["accuracy"], label="Training")
    ax.plot(history.history["val_accuracy"], label="Validation")
    ax.set(title=name, xlabel="Epoch index (starting at 0)", ylabel="Accuracy")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 12. เลือกโมเดลเดิมจาก Validation Accuracy
เปรียบเทียบ Validation Accuracy ของ Epoch สุดท้ายของ DNN และ CNN ด้วยเงื่อนไขเดียว กรณีเท่ากันเลือก CNN ทั้งสองโมเดลเดิมเก็บน้ำหนักของ Epoch สุดท้าย จึงใช้ History ได้


In [ ]:
dnn_val_acc = history_dnn_original.history["val_accuracy"][-1]
cnn_val_acc = history_cnn_original.history["val_accuracy"][-1]

if cnn_val_acc >= dnn_val_acc:
    use_cnn = True
    baseline = cnn_original
    baseline_history = history_cnn_original
    baseline_name = "CNN Original"
else:
    use_cnn = False
    baseline = dnn_original
    baseline_history = history_dnn_original
    baseline_name = "DNN Original"

print("DNN Validation Accuracy:", dnn_val_acc)
print("CNN Validation Accuracy:", cnn_val_acc)
print("เลือกโมเดล:", baseline_name)


## 13. ตั้งสมมติฐานและสร้าง โมเดลใหม่ (เพิ่ม Dropout + L2 + Early Stopping) แยกจาก โมเดลเดิมที่เลือก
เราจะทดลองสามเทคนิค โดยคงข้อมูลเดิมและโครงสร้างหลักไว้:

| เทคนิค | เปลี่ยนอะไร | เหตุผลที่ทดลอง |
|---|---|---|
| Dropout 0.3 | สุ่มปิดผลลัพธ์บางหน่วยเฉพาะตอนฝึก | ลดการพึ่งพาหน่วยบางตัวมากเกินไป เมื่อผล Training ดีขึ้นแต่ Validation ตามไม่ทัน |
| L2 (`l2(0.001)`) | เพิ่มค่าปรับตามกำลังสองของน้ำหนักเข้าไปใน Loss | จำกัดน้ำหนักขนาดใหญ่ เพื่อลดการปรับเข้ากับข้อมูลฝึกมากเกินไป |
| Early Stopping | หยุดเมื่อ Validation Accuracy ไม่ดีขึ้นตามเงื่อนไข | ลดการฝึกต่อที่ไม่เพิ่มคะแนน Validation และคืนค่าน้ำหนักที่ดีที่สุด |

ตรวจกราฟข้อ 11 ก่อนเขียนเหตุผลจริง ถ้าไม่มีสัญญาณ Overfitting อย่าเขียนว่าเกิดแน่นอน เทคนิคเหล่านี้เป็นการทดลอง ไม่รับประกันว่าดีขึ้น และเมื่อรวมเทคนิคทั้งสามในรอบเดียว จะยังแยกไม่ได้ว่าเทคนิคใดช่วยมากน้อยเพียงใด

`new_model` เป็นโมเดลใหม่ เริ่มเรียนรู้น้ำหนักใหม่ ไม่ใช่การนำ โมเดลเดิมที่เลือก มาฝึกต่อ `Dropout(0.3)` ไม่ได้ลบ Neurons ถาวร และปิดการสุ่มเมื่อประเมิน/ทำนาย ส่วน L2 เพิ่ม Penalty ใน Loss โดยใช้ค่าเริ่มทดลอง `0.001`


In [ ]:
l2_regularizer = tf.keras.regularizers.l2(0.001)
print("L2 coefficient:", 0.001)

if use_cnn:
    x_train_selected, x_val_selected, x_test_selected = x_train_cnn, x_val_cnn, x_test_cnn
    new_model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation="relu", kernel_regularizer=l2_regularizer),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", kernel_regularizer=l2_regularizer),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(64, activation="relu", kernel_regularizer=l2_regularizer),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ], name="cnn_new")
else:
    x_train_selected, x_val_selected, x_test_selected = x_train, x_val, x_test
    new_model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation="relu", kernel_regularizer=l2_regularizer),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu", kernel_regularizer=l2_regularizer),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ], name="dnn_new")

new_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
new_model.summary()


## 14. ฝึกโมเดลใหม่
Early Stopping ติดตาม Validation Accuracy หากไม่ดีขึ้น 3 Epoch ติดต่อกันจะหยุดและคืนค่าน้ำหนักที่ดีที่สุด อาจฝึกครบ 10 Epoch ได้


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=3,
    restore_best_weights=True
)

history_new_model = new_model.fit(
    x_train_selected,
    y_train,
    validation_data=(x_val_selected, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

print(
    "Actual epochs:",
    len(history_new_model.history["loss"])
)


## 15. เปรียบเทียบกราฟ โมเดลเดิมที่เลือก กับ โมเดลใหม่ (เพิ่ม Dropout + L2 + Early Stopping)
ดู Validation Loss และ Accuracy ร่วมกัน อย่าตัดสินจากช่องว่างเล็กลงอย่างเดียว เพราะทั้งสองชุดอาจแย่ลงได้ Dropout ทำงานเฉพาะตอนฝึก จึงอาจเห็น Training Accuracy ต่ำกว่า Validation ได้


In [ ]:
# กราฟ Loss ของโมเดลเดิมที่เลือกและโมเดลใหม่
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, history in zip(axes, ["Baseline", "New"], [baseline_history, history_new_model]):
    epochs = range(1, len(history.history["loss"]) + 1)
    ax.plot(epochs, history.history["loss"], label="Training Loss")
    ax.plot(epochs, history.history["val_loss"], label="Validation Loss")
    ax.set(title=name, xlabel="Epoch", ylabel="Loss")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 16. เลือกโมเดลสุดท้ายจาก Validation
ประเมินโมเดลใหม่หลังคืนค่าน้ำหนักจาก Early Stopping แล้วเปรียบเทียบกับโมเดลเดิมที่เลือก ก่อนประเมิน Test

Loss ของโมเดลใหม่รวมค่า L2 penalty จึงควรพิจารณา Accuracy และกราฟร่วมด้วย


In [ ]:
_, new_val_acc = new_model.evaluate(x_val_selected, y_val, verbose=0)
baseline_val_acc = baseline_history.history["val_accuracy"][-1]

if new_val_acc > baseline_val_acc:
    final_model = new_model
    final_name = "New"
else:
    final_model = baseline
    final_name = baseline_name

print("Baseline Validation Accuracy:", baseline_val_acc)
print("New Validation Accuracy:", new_val_acc)
print("โมเดลสุดท้าย:", final_name)


## 17. ตารางเปรียบเทียบผล
ประเมิน Test หลังเลือกโมเดลสุดท้ายแล้ว โจทย์ขอผล Test ของ DNN/CNN และ Baseline/Improved ด้วย จึงรายงานแต่ละโมเดลในรอบสุดท้ายโดยไม่ย้อนกลับไปปรับโมเดล หากอาจารย์กำหนดให้ประเมิน Test เฉพาะตัวสุดท้าย ให้ยึดตามคำชี้แจงของอาจารย์

ตาราง DNN/CNN ใช้ Training Accuracy จาก History ส่วนตาราง Baseline/Improved ประเมินโมเดลทั้งคู่หลังฝึกเสร็จ โดย Dropout ปิดขณะประเมิน


In [ ]:
# ประเมิน Test ครั้งเดียวต่อโมเดลหลังเลือกโมเดลสุดท้ายแล้ว
dnn_test_loss, dnn_test_acc = dnn_original.evaluate(x_test, y_test, verbose=0)
cnn_test_loss, cnn_test_acc = cnn_original.evaluate(x_test_cnn, y_test, verbose=0)
new_test_loss, new_test_acc = new_model.evaluate(x_test_selected, y_test, verbose=0)

# ตาราง DNN และ CNN ตามโจทย์หลัก
display(pd.DataFrame({
    "Model": ["DNN Original", "CNN Original"],
    "Training Accuracy": [history_dnn_original.history["accuracy"][-1], history_cnn_original.history["accuracy"][-1]],
    "Validation Accuracy": [dnn_val_acc, cnn_val_acc],
    "Test Accuracy": [dnn_test_acc, cnn_test_acc],
    "Test Loss": [dnn_test_loss, cnn_test_loss]
}).round(4))

# โมเดลเดิมที่เลือกคือ DNN หรือ CNN จึงนำผล Test ข้างต้นมาใช้ซ้ำ
if use_cnn:
    baseline_test_acc, baseline_test_loss = cnn_test_acc, cnn_test_loss
else:
    baseline_test_acc, baseline_test_loss = dnn_test_acc, dnn_test_loss
_, baseline_train_acc = baseline.evaluate(x_train_selected, y_train, verbose=0)
_, new_train_acc = new_model.evaluate(x_train_selected, y_train, verbose=0)

display(pd.DataFrame({
    "Metric": ["Training Accuracy", "Validation Accuracy", "Test Accuracy", "Test Loss", "Actual Epochs"],
    "Baseline": [baseline_train_acc, baseline_val_acc, baseline_test_acc, baseline_test_loss, len(baseline_history.history["loss"])],
    "Improved": [new_train_acc, new_val_acc, new_test_acc, new_test_loss, len(history_new_model.history["loss"])]
}).round(4))


## 18. ดูว่า DNN และ CNN สับสนคลาสไหน
`predict()` ให้คะแนน 10 คลาสต่อภาพ `argmax(axis=1)` เลือกเลขคลาสที่คะแนนสูงที่สุดในแต่ละแถว

Confusion Matrix: แถว = คำตอบจริง คอลัมน์ = คำทำนาย แนวทแยง = ถูก นอกแนวทแยง = ผิด เช่น แถว Shirt คอลัมน์ T-shirt/top หมายถึง Shirt ถูกทำนายเป็น T-shirt/top


In [ ]:
# ทำนาย Test Set เพื่อสร้าง Confusion Matrix ของ DNN และ CNN
dnn_pred = np.argmax(dnn_original.predict(x_test, verbose=0), axis=1)
cnn_pred = np.argmax(cnn_original.predict(x_test_cnn, verbose=0), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, name, y_pred in zip(axes, ["DNN Original", "CNN Original"], [dnn_pred, cnn_pred]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, labels=np.arange(10), display_labels=class_names,
        cmap="Blues", xticks_rotation=90, colorbar=False, ax=ax
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
# ผลทำนาย dnn_pred และ cnn_pred นำไปใช้เลือกภาพถูกและผิดในขั้นตอนถัดไป


## 19. แสดงภาพถูก 5 ภาพและผิด 5 ภาพ
เลือก DNN หรือ CNN ที่ Test Accuracy สูงกว่าเพื่อแสดงตัวอย่างตามขั้นตอนงานเดิม การเลือกส่วนนี้ไม่เปลี่ยนโมเดลสุดท้ายที่เลือกด้วย Validation

`np.where()` หาตำแหน่งที่ทำนายตรง/ไม่ตรงกับคำตอบจริง แล้ว `[:5]` เลือกห้าภาพแรก ตัวอย่างเหล่านี้ไม่ได้แทนความผิดพลาดทั้งหมด


In [ ]:
if cnn_test_acc >= dnn_test_acc:
    example_name, y_pred = "CNN Original", cnn_pred
else:
    example_name, y_pred = "DNN Original", dnn_pred

correct_indices = np.where(y_pred == y_test)[0][:5]
wrong_indices = np.where(y_pred != y_test)[0][:5]

print("Model used for examples:", example_name)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for row, indices in enumerate(
    [correct_indices, wrong_indices]
):
    for col in range(5):
        ax = axes[row, col]
        ax.axis("off")

        if col >= len(indices):
            continue

        idx = indices[col]

        ax.imshow(x_test[idx], cmap="gray")
        ax.set_title(
            f"True: {class_names[y_test[idx]]}\n"
            f"Pred: {class_names[y_pred[idx]]}",
            color="green" if row == 0 else "red",
            fontsize=9
        )

plt.suptitle(example_name)
plt.tight_layout()
plt.show()


## 20. เขียนอภิปรายจากผลจริงหลังรัน

| คำถาม | หลักฐานที่จะอ้าง |
|---|---|
| ทำไมเลือก โมเดลเดิมที่เลือก ตัวนี้? | ค่า Validation Accuracy ข้อ 12 |
| ปรับอะไรและเพราะอะไร? | กราฟข้อ 11 และสมมติฐานข้อ 13 |
| โมเดลใหม่ (เพิ่ม Dropout + L2 + Early Stopping) ดีขึ้นหรือไม่? | ผล Validation ข้อ 16 และตารางข้อ 17 |
| ลด Overfitting หรือไม่? | แนวโน้ม Training/Validation Loss ข้อ 15 |
| โมเดลสุดท้ายคืออะไรและ Test ได้เท่าไร? | การเลือกข้อ 16 และผลข้อ 17 |
| โมเดลสับสนอะไร? | Confusion Matrix ข้อ 18 |
| ภาพที่ผิดมีลักษณะอย่างไร? | ตัวอย่างภาพข้อ 19 |

เติมข้อสรุปของตนเองหลังรัน: **[โมเดลเดิมที่เลือก และเหตุผล] → [เทคนิคที่ทดลอง] → [ผลต่างที่ได้จริง] → [โมเดลสุดท้ายและ Test] → [ข้อผิดพลาดที่ยังพบ]**

อย่าสรุปว่าเทคนิคช่วยแน่นอนจากชื่อ โมเดลใหม่ (เพิ่ม Dropout + L2 + Early Stopping) หรือจาก Training Accuracy อย่างเดียว ผลการรันครั้งเดียวอาจมีความแปรปรวนจากการเริ่มน้ำหนักและการสุ่ม และภาพ Fashion-MNIST ขนาดเล็กสีเทายังไม่ใช่หลักฐานว่าใช้กับภาพสินค้าจริงได้ทันที
